## Set things up

### Import libraries
Add directory with the CIBUSmod to path and import CIBUSmod and other packages for handling data and plotting

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(),'..'))

import CIBUSmod as cm
import CIBUSmod.utils.plot as plot

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


 -----------------------
|   Imported CIBUSmod   |
 -----------------------
commit : 85b0ac0e2e1065ecbb01c64ddd5763db4f441466
branch : main
dirty  : True
remote : https://github.com/SLU-foodsystems/CIBUSmod
    


### Set up session
Instantiate a `Session` with a `name` and `data_path` to the folder with input data. Next add scenarios to run with `.add_scenario()`, in this case we only use default data by setting `scenario_workbooks` to `None`

In [2]:
# Create session
session = cm.Session(
    name = 'out',
    data_path = '../data'
)

# Add scenarios
session.add_scenario(
    name = 'base year',
    scenario_workbooks = None,
    years = '2020'
)

# Add scenarios
session.add_scenario(
    name = 'base year (x0 crop areas)',
    scenario_workbooks = None,
    years = '2020'
)

session.add_scenario(
    name = 'base year (sp=0.2)',
    scenario_workbooks = None,
    years = '2020'
)

session.add_scenario(
    name = 'base year (sp=0.8)',
    scenario_workbooks = None,
    years = '2020'
)

A scenario with the name 'base year' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'base year (x0 crop areas)' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'base year (sp=0.2)' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'base year (sp=0.8)' already exists use .update_scenario() or .remove_scenario() instead.


## Run baseline calculations

In [3]:
%%time
cm.ParameterRetriever.update_relation_tables()
# Instatiate Regions
regions = cm.Regions(
    par = cm.ParameterRetriever('Regions')
)

# Instantiate DemandAndConversions
demand = cm.DemandAndConversions(
    par = cm.ParameterRetriever('DemandAndConversions')
)

# Instantiate CropProduction
crops = cm.CropProduction(
    par = cm.ParameterRetriever('CropProduction'),
    regions = regions
)    

# Instantiate AnimalHerds
# Each AnimalHerd object is stored in an indexed pandas.Series
herds = cm.make_herds(regions)

# Instantiate WasteAndCircularity
waste = cm.WasteAndCircularity(
    demand = demand,
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('WasteAndCircularity')
)

# Instantiate feed management
feed_mgmt = cm.FeedMgmt(
    herds = herds,
    par = cm.ParameterRetriever('FeedMgmt')
)

# Instantiate by-product management
byprod_mgmt = cm.ByProductMgmt(
    demand = demand,
    herds = herds,
    par = cm.ParameterRetriever('ByProductMgmt')
)

# Instantiate manure management
manure_mgmt = cm.ManureMgmt(
    herds = herds,
    feed_mgmt = feed_mgmt,
    par = cm.ParameterRetriever('ManureMgmt'),
    settings = {
        'NPK_excretion_from_balance' : True
    }
)

# Instantiate crop residue managment
crop_residue_mgmt = cm.CropResidueMgmt(
    demand = demand,
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('CropResidueMgmt')
)

# Instantiate cover crops management
cover_crops_mgmt = cm.CoverCropsMgmt(
    crops = crops,
    par = cm.ParameterRetriever('CoverCropsMgmt')
)

# Instantiate plant nutrient management
plant_nutrient_mgmt = cm.PlantNutrientMgmt(
    demand = demand,
    regions = regions,
    crops = crops,
    waste = waste,
    herds = herds,
    cover_crops_mgmt = cover_crops_mgmt,
    par = cm.ParameterRetriever('PlantNutrientMgmt')
)

# Instatiate machinery and energy management
machinery_and_energy_mgmt  = cm.MachineryAndEnergyMgmt(
    regions = regions,
    crops = crops,
    waste = waste,
    herds = herds,
    par = cm.ParameterRetriever('MachineryAndEnergyMgmt')
)

# Instatiate inputs management
inputs = cm.InputsMgmt(
    demand = demand,
    crops = crops,
    waste = waste,
    herds = herds,
    par = cm.ParameterRetriever('InputsMgmt')
)

# Instantiate geo distributor
geodist = cm.GeoDistributor(
    regions = regions,
    demand = demand,
    crops = crops,
    herds = herds,
    feed_mgmt = feed_mgmt,
    par = cm.ParameterRetriever('GeoDistributor')
)

CPU times: total: 13.5 s
Wall time: 14.4 s


In [ ]:
%%time
# Set to true to print progress messages
msg = True
scale_powers = {
    'base year (sp=0.2)' : 0.2,
    'base year (sp=0.8)' : 0.8,
}

# Loop through scenarios and years
for scn, year in session.iterate('all'):
    print(scn,year)
    # Reload parameter data
    cm.ParameterRetriever.update_all_parameter_values(
        **session[scn],
        year=year
    )

    # Get region attributes
    regions.calculate(verbose=msg)

    # Calculate food demand
    demand.calculate(verbose=msg)

    # Calculate crops
    crops.calculate(
        verbose=msg
    )

    # Calculate herds
    for h in herds:
        h.calculate(verbose=msg)

    # Induce beef exports in DemandAndConversions if beef production from dairy
    # systems under given demand for milk products exceeds total beef demand.
    # This is to avoid not finding any solution when running the GeoDistributor.
    cm.helpers.induce_beef_exports(
        demand = demand,
        herds = herds
    )
    

    # Calculate feed
    feed_mgmt.calculate(verbose=msg)

    # Distribute animals and crops
    # Make optimisation problem
    geodist.make(use_cons=[1,2,3,4,5,6,7], scale_power=scale_powers.get(scn, 0.4), verbose=msg)
    # Solve optimisation problem (move to next scn/year if it fails)
    try:
        geodist.solve(verbose=msg, apply_solution=False)
    except Exception as e:
        print(f'(!!!) GeoDistributor failed for {scn}, {year} with {type(e).__name__}: {e}')
        continue

    if 'x0 crop areas' in scn:
        # Set crop areas to x0 areas (for validation)
        geodist.x['crp'] = geodist.x0['crp']

    # Apply solution
    geodist.apply_solution()
        
    # Redistribute feeds (not yet implemented) and calculate enteric CH4 emissions
    feed_mgmt.calculate2(verbose=msg)

    # Balance by-product demand and suply
    byprod_mgmt.calculate(verbose=msg)

    # Calculate manure
    manure_mgmt.calculate(verbose=msg)

    # Calculate harvest of crop residues
    crop_residue_mgmt.calculate(verbose=msg)

    # Calculate cover crop areas
    cover_crops_mgmt.calculate(verbose=msg)

    # Calculate treatment of wastes and other feedstocks
    waste.calculate(verbose=True)

    # Calculate plant nutrient management
    plant_nutrient_mgmt.calculate(verbose=msg)

    # Calculate energy requirements
    machinery_and_energy_mgmt.calculate(verbose=msg)

    # Calculate inputs supply chain emissions
    inputs.calculate(verbose=msg)

    # Store results
    session.store(
        scn, year,
        demand, regions, crops, herds, waste, geodist
    )
